# 07 · Final pipeline

Starts from the best detector of notebooks 05–06 under the benchmark protocol (`BASE`) and changes
**one thing at a time**. The two required parts are kept separate:

* **A. Data and inference strategy.** The architecture and training recipe stay those of `BASE`.
  * A1: inference only, on the `BASE` checkpoint: flip TTA, multi-scale TTA, tiled inference.
  * A2: training data: denser frame sampling, ignore regions left visible, down-scaling
    augmentation, 2×2 tiles (trained and predicted on tiles).
* **B. Model.** Data and inference stay at the best setting from A: a larger backbone, a longer
  schedule, and a YOLO + RT-DETR ensemble (weighted boxes fusion).

The best 2 candidates are then **re-trained on folds 1 and 2** (out-of-fold check). The two final
submissions are chosen by mean OOF mAP, not by the public leaderboard. Finally the selected pipeline is
trained on all 63 labelled clips and predicts the test set.

**Kaggle inputs:** competition data, the output of notebook 02 (`prepared/`, `prepared_nomask/`), and
the outputs of notebooks 05 and 06 (checkpoints of `BASE` and the best RT-DETR run).

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "ivan-in-a-nutshell/AI_HW_2"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Data and the fixed benchmark protocol: fold 0, every 10th frame, unmasked test-scale validation frames
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from visdet.data import find_raw_root, find_prepared, read_prepared, fold_split
from visdet.evaluate import build_gt
from visdet.plotting import setup_style, save, SERIES
from visdet.results import load_log, family_table, best_run, plot_histories, plot_breakdown, styled

RAW, PREP = find_raw_root(), find_prepared()
RUNS = Path("/kaggle/working/runs") if ON_KAGGLE else REPO / "runs"
FOLD, STEP = 0, 10
images, boxes, folds = read_prepared(PREP)
train_rows, val_rows = fold_split(images, folds, FOLD, train_step=STEP, val_step=STEP)
gt = build_gt(val_rows, boxes)
N_GPU = torch.cuda.device_count()
SMOKE = os.environ.get("VISDET_SMOKE") == "1"   # CPU dry run: 1 epoch on a few frames, nothing logged for real
if SMOKE:
    val_rows = val_rows.head(8); gt = build_gt(val_rows, boxes)
setup_style()
print(f"fold {FOLD}: {len(train_rows)} training frames, {len(val_rows)} validation frames; {N_GPU} GPU(s)")

In [ ]:
# Pipeline context: the starting point is the best YOLO run of notebook 05
from visdet.pipeline import Spec, Pipeline, oof_table
from visdet.data import find_prepared
log = load_log()
YOLO, DETR = ['yolo11s_base', 'yolo11s_960', 'yolo11s_1536', 'yolo11s_p2', 'yolo11s_tal'], ['rtdetr_base', 'rtdetr_q500', 'rtdetr_matcher', 'rtdetr_bblr']
BASE = best_run(log, YOLO)
DEVICE = ",".join(str(i) for i in range(N_GPU)) or "cpu"
TRAIN_KW = {"device": DEVICE, "workers": 4, "verbose": False}
UL_SMOKE = {"imgsz": 320, "epochs": 1, "time": None, "batch": 4, "fraction": 0.005, "workers": 0} if SMOKE else {}
try:
    PREP_NOMASK = find_prepared("prepared_nomask")
except FileNotFoundError:
    PREP_NOMASK = None
pipe = Pipeline(RAW, PREP, RUNS, images, boxes, folds, BASE, TRAIN_KW, PREP_NOMASK, UL_SMOKE)
base_cfg = pipe.base_cfg
print(f"BASE = {BASE}: fold-0 mAP {log.loc[BASE, 'mAP']:.4f}")

## A1. Inference strategy (same checkpoint, no training)

| run | inference |
|---|---|
| `BASE+tta` | original + horizontally flipped frame, fused with WBF |
| `BASE+tta_ms` | flip TTA at the training size and at 1.25× the size |
| `BASE+tiles` | 2×2 overlapping tiles + the full frame, class-wise NMS (the model never saw tiles) |

In [ ]:
# Evaluate the inference variants of the BASE checkpoint
base = Spec(BASE, weights_from=BASE)
A_INFER = [base.but(f"{BASE}+tta", infer="tta"), base.but(f"{BASE}+tta_ms", infer="tta_ms"),
           base.but(f"{BASE}+tiles", infer="tiles")]
for spec in A_INFER:
    pipe.evaluate(spec)

## A2. Training data (same model and budget, retrained)

| run | data change | motivation |
|---|---|---|
| `pipe_step5` | every 5th frame instead of every 10th (2× frames, same GPU time) | more scene diversity per GPU-hour (EDA §6: IoU 0.59 at 5 frames) |
| `pipe_nomask` | ignore regions left visible (their boxes still dropped) | tests the gray fill: without it, unlabelled crowds are taught as background |
| `pipe_degrade` | random ×0.5–0.8 down/up-scaling (p = 0.25) | 16 % of test frames are stored at quarter resolution (EDA §1) |
| `pipe_tiles` | train on 2×2 overlapping tiles, predict on tiles + full frame | objects ~1.8× larger at the same input size |

In [ ]:
# Retrain BASE with one data change each (2 GPU-hours per run)
patch = base_cfg.get("patch") or {}
A_DATA = [
    Spec("pipe_step5", train_step=5),
    Spec("pipe_nomask", masked=False),
    Spec("pipe_degrade", changes={"patch": patch | {"degrade_p": 0.25, "degrade_range": [0.5, 0.8]}}),
    Spec("pipe_tiles", tiles=True),
]
for spec in A_DATA:
    if spec.masked or PREP_NOMASK is not None:
        pipe.evaluate(spec)

In [ ]:
# Part A results (ΔmAP against BASE)
log = load_log()
styled(family_table(log, [BASE] + [s.name for s in A_INFER + A_DATA]))

## B. Model (data and inference from the best part-A setting)

In [ ]:
# Best data setting from A2, then one model change at a time on top of it
log = load_log()
best_data = best_run(log, [BASE] + [s.name for s in A_DATA])
data_spec = next((s for s in A_DATA if s.name == best_data), Spec(BASE))
bigger = {k: v.replace("11s", "11m") for k, v in base_cfg.items() if k in ("model", "weights") and isinstance(v, str)}
B_MODEL = [
    data_spec.but("pipe_m", changes=bigger | {"batch": base_cfg["batch"] // 2}),          # YOLO11m, same budget
    data_spec.but("pipe_long", changes={"time": 2 * (base_cfg.get("time") or 1.0)}),      # 2x the budget
]
for spec in B_MODEL:
    pipe.evaluate(spec)
# ensemble of the best YOLO-type run so far and the best RT-DETR run (needs notebook 06 output)
detr_runs = [r for r in DETR if r in log.index]
if detr_runs:
    best_yolo = best_run(load_log(), [best_data] + [s.name for s in B_MODEL])
    ENS = Spec("pipe_ens", ensemble=[best_yolo, best_run(log, detr_runs)])
    pipe.evaluate(ENS)
print("best data setting:", best_data)

In [ ]:
# Part B results (ΔmAP against the best part-A run)
log = load_log()
styled(family_table(log, [best_data] + [s.name for s in B_MODEL] + ["pipe_ens"]))

## C. Out-of-fold check of the finalists

Fold 0 picked the settings above. To avoid overfitting that choice to 13 clips, the best two trained
pipelines are re-trained on folds 1 and 2 (different held-out clips) under the same budget.

In [ ]:
# Train + evaluate the two best trained specs on folds 1 and 2
log = load_log()
trained = {s.name: s for s in [Spec(BASE)] + A_DATA + B_MODEL}
ranked = log.loc[[n for n in trained if n in log.index], "mAP"].sort_values(ascending=False).index[:2]
FINALISTS = [trained[n] for n in ranked]
for spec in FINALISTS:
    for k in (1, 2):
        pipe.evaluate(spec, fold=k)
oof = oof_table(load_log(), [s.name for s in FINALISTS])
oof.round(4)

## D. Final models and submissions

Both finalists are re-trained on **all 63 labelled clips** (`fold = -1`, 1.5× the budget for the larger
training set) and predict the original test frames. Submission 1 = best mean OOF mAP, submission 2 = the
runner-up. Seeds and settings are in each run's `config.yaml` under `experiments/runs/`.

In [ ]:
# Train on all clips and write the two final submissions
OUT = Path("/kaggle/working") if ON_KAGGLE else REPO
for rank, name in enumerate(oof.index, start=1):
    budget = (base_cfg | trained[name].changes).get("time") or 1.0               # this spec's validated budget
    spec = trained[name].but(name, changes={"time": 1.5 * budget})
    sub = pipe.submission(spec, fold=-1, limit=16 if SMOKE else None)
    sub.to_csv(OUT / f"submission_{rank}_{name}.csv", index=False)
    print(f"submission {rank}: {name} (OOF mAP {oof.loc[name, 'mean']:.4f} ± {oof.loc[name, 'std']:.4f})")

## Findings

_Fill in after the runs:_ which data/inference changes generalised (A), which model changes added on top
(B), the OOF table (are the fold-0 gains stable across folds?), the public LB of both submissions, and
the unsuccessful attempts with a plausible reason.

**After the session:** download `/kaggle/working/repo/experiments/` (metrics, per-epoch histories) and
`images/` from the notebook output and commit them, so later notebooks and the report can use them.
Checkpoints stay in the notebook output (`/kaggle/working/runs`).